# Практика § 5.4: температура в двух опытах

Сначала исследуйте распределение для фиксированной модели, затем отдельную таблицу уверенности. Случайный выбор не является проверкой фактов.

## Доли и случайный выбор

Запустите код с одинаковым зерном и сравните частоты в большем числе повторений.

In [ ]:
import numpy as np
rng=np.random.default_rng(42)
logits=np.array([2.,1.,0.])
def probs(t):
    z=logits/t
    w=np.exp(z-z.max())
    return w/w.sum()
for t in [.5,1.,2.]:
    p=probs(t)
    draws=rng.choice(3,size=10000,p=p)
    print(t,p.round(3),np.bincount(draws,minlength=3)/10000)

## Уверенность и ответы

Это маленькая учебная таблица; на четырёх строках нельзя надёжно калибровать настоящую модель.

In [ ]:
confidence=np.array([.9,.8,.7,.6])
correct=np.array([1,0,1,0])
for lo,hi in [(.5,.75),(.75,1.01)]:
    mask=(confidence>=lo)&(confidence<hi)
    print(lo,hi,'число',mask.sum(),'средняя уверенность',confidence[mask].mean(),'частота верных',correct[mask].mean())

## Порядок классов

Проверьте, что масштабирование всех логитов положительной температурой не меняет победителя.

In [ ]:
scores=np.array([[2.,0.,1.],[0.,3.,1.]])
for t in [.5,1.,2.,10.]:
    print(t,np.argmax(scores/t,axis=1))

## Самостоятельно

Объясните, почему подбор температуры на этих четырёх ответах и отчёт о качестве на них же был бы нечестной проверкой.

## Самопроверка




### 1. Что делает температура T>1 с распределением для разных логитов?

A. Обычно делает его ровнее

B. Меняет порядок логитов

C. Убирает знаменатель softmax

D. Меняет веса модели




### 2. Что делает температура 0<T<1?

A. Обычно усиливает лидера

B. Всегда выбирает последний класс

C. Меняет число классов

D. Стирает логиты




### 3. Может ли общая положительная температура поменять лучший класс?

A. Да, всегда

B. Нет, порядок логитов сохраняется

C. Только при T=2

D. Только для текста




### 4. Почему нельзя подставить T=0 в формулу?

A. В ней есть деление на T

B. Тогда нет классов

C. Потеря станет MSE

D. Логиты исчезнут




### 5. Когда температура влияет на разнообразие генерации?

A. Когда токен выбирают случайно по распределению

B. При неизменном жадном выборе

C. При подсчёте параметров

D. При сохранении JPEG




### 6. Что означает калибровка уверенности?

A. Соответствие заявленной уверенности частоте верных ответов

B. Всегда правильный один ответ

C. Минимальную обучающую потерю

D. Равные логиты всех классов




### 7. Чему равна средняя уверенность чисел 0,9;0,8;0,7;0,6?

A. 0,5

B. 0,6

C. 0,75

D. 1




### 8. Где подбирают температуру калибровки?

A. На итоговом тесте

B. На отдельной проверочной части

C. На одном лучшем примере

D. В исходном коде без данных




### 9. Может ли калибровка исправить ошибочный порядок классов?

A. Да, если T больше 2

B. Нет, общая положительная T сохраняет порядок

C. Только на русском языке

D. Всегда при T<1




### 10. Почему оценка 0,9 не доказывает истинность фразы генератора?

A. Это числовая оценка модели, а факты требуют внешней проверки

B. Потому что 0,9 меньше 1

C. Потому что softmax не работает для текста

D. Потому что память мала




## Подсказки и решения

### Вопрос 1

**Подсказка**


Логиты делят на число больше единицы.



**Решение**


**A.** Разницы масштабированных логитов меньше, доли обычно распределяются ровнее.



### Вопрос 2

**Подсказка**


Деление на малое число увеличивает разницы.



**Решение**


**A.** Распределение становится резче при разных логитах.



### Вопрос 3

**Подсказка**


Положительное деление сохраняет порядок чисел.



**Решение**


**B.** Победитель argmax остаётся прежним при T>0.



### Вопрос 4

**Подсказка**


Посмотрите на z_i/T.



**Решение**


**A.** Ноль в знаменателе не определён; в генераторах это отдельное правило жадного выбора.



### Вопрос 5

**Подсказка**


Изменяются доли вариантов.



**Решение**


**A.** Случайный выбор чаще или реже берёт альтернативы в зависимости от T.



### Вопрос 6

**Подсказка**


Сравните ответы с уверенностью около 0,8.



**Решение**


**A.** Хорошая калибровка означает около 80% верных среди многих ответов с такой уверенностью.



### Вопрос 7

**Подсказка**


Сложите и разделите на четыре.



**Решение**


**C.** Сумма 3,0 делится на 4 и даёт 0,75.



### Вопрос 8

**Подсказка**


Итоговые данные нельзя использовать для настройки.



**Решение**


**B.** Температуру выбирают на проверочной части, итог измеряют отдельно.



### Вопрос 9

**Подсказка**


Сравните argmax до и после деления.



**Решение**


**B.** Одна температура изменяет доли, а не выбранный класс.



### Вопрос 10

**Подсказка**


Откуда модель получает правильность факта?



**Решение**


**A.** Оценка токена не подтверждает фактическое утверждение; нужен источник и проверка.



